In [0]:
import pyspark
from pyspark.sql.types import *
import pyspark.sql.functions as f
from pyspark.sql import SparkSession, Row, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.functions import when
from typing import Union, Optional, List
from dataclasses import dataclass
from pyspark.sql.types import IntegerType
from functools import reduce
from datetime import timedelta
from pyspark.sql.functions import broadcast
from pyspark.sql.functions import when, lit

#import
#from pls_common_data_store import pls_data_store
#pds = pls_data_store()

#ignore strange depreciation warnings
from warnings import simplefilter 
simplefilter(action='ignore', category=DeprecationWarning)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.shuffle.partitions","auto")
spark.conf.get("spark.sql.shuffle.partitions")


In [0]:
# Data Locations
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')
mda = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/dashboard/campaign/version=v2/source=azure')
points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
mmoi= spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/MEDIA_MEAS_OFFER_INFO')
new_th = spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/TARGET_HISTORY')
old_th = spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/bullseye/TARGET_HISTORY_FULL_20251015/').withColumnRenamed('ehhn', 'hshd_code').select('TARGET_ID', 'HSHD_CODE', 'PRIORITY', 'TEST_CONTROL_ID', 'OFFER_ID', 'DECILE', 'SCORE')
target_history = new_th.union(old_th)
mhtv = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped')
redemptions = spark.read.parquet(f'abfss://acds@sa8451posprd.dfs.core.windows.net/transaction_coupon_fct')
downloads = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/intermediate/engagements/coupon_downloads/')

#### Pulling Audio and CTV from Media Hist Remastered

In [0]:
media_history_v2 = spark.read.parquet(f'abfss://mi-adhoc@sa8451midsrdev.dfs.core.windows.net/media_hist_remastered')
media_history_v2.display()

In [0]:
mhv = spark.read.parquet(f'abfss://mi-adhoc@sa8451midsrdev.dfs.core.windows.net/media_hist_remastered').filter(f.col('campaign_type') == 'AUDIO').filter(f.col('modality') == 'all').filter(f.col('market') == 'kroger_only').filter(f.col('manufacturer') == 'Kroger Personal Finance')
mhv.display()

In [0]:
# filtering to CTV or Audio
media_history_v2_ctv_audio = media_history_v2.filter(
    (f.col("manufacturer") == "Kroger Personal Finance") &
    (f.col("campaign_type").isin(["AUDIO", "CTV"])) & 
    (f.col("modality") == "all") &
    (f.col("market") == "kroger_only") & 
    (f.col("camp_start_date") >= "2026-01-01") & 
    (f.col("adjusted_top_performer") == "Top-Performer_KRO")
)

# Add year and quarter columns based on camp_start_date
media_history_v2_ctv_audio = media_history_v2_ctv_audio.withColumn(
    "year", f.year(f.col("camp_start_date"))
).withColumn(
    "quarter", f.quarter(f.col("camp_start_date"))
)

media_history_v2_ctv_audio.display()

In [0]:
# Aggregation of upstream data calculations (I think for the most part this is not needed, if there is only 1 row per top performing ctv/aduio)

media_history_v2_ctv_audio_agg = media_history_v2_ctv_audio.groupby(
    "kpm_project_id", "job_id", "project_name", "campaign_type", "quarter", "year", "manufacturer", "camp_start_date", "camp_end_date").agg(
    # Campaign Detail Metrics
    f.max("test_hh_count").alias("HHs_reached"),
    f.avg("camp_cost").alias("Investment"),
    f.avg("camp_cost").alias("camp_cost"),
    f.avg("total_redemptions_cost").alias("Redemptions_Cost"),

    # Closed Loop Metrics
    f.avg("sales_test_total").alias("sales_test_total"),
    f.avg("sales_cont_total").alias("sales_cont_total"),
    f.avg("sales_uplift_total").alias("sales_uplift_total"),
    f.avg("sales_uplift_per_hh").alias("sales_uplift_per_hh"),

    f.avg("hhpen_test_total").alias("hhpen_test_total"),
    f.avg("hhpen_cont_total").alias("hhpen_cont_total"),
    f.avg("hhpen_uplift_total").alias("hhpen_uplift_total"),
    f.avg("hhpen_uplift_per_hh").alias("hhpen_uplift_per_hh"),

    f.avg("visits_test_total").alias("visits_test_total"),
    f.avg("visits_cont_total").alias("visits_cont_total"),
    f.avg("visits_uplift_total").alias("visits_uplift_total"),
    f.avg("visits_uplift_per_hh").alias("visits_uplift_per_hh"),

    f.avg("units_test_total").alias("units_test_total"),
    f.avg("units_cont_total").alias("units_cont_total"),
    f.avg("units_uplift_total").alias("units_uplift_total"),
    f.avg("units_uplift_per_hh").alias("units_uplift_per_hh"),

    # Engagement Metrics
    f.avg("clickthrough_rate").alias("clickthrough_rate"),
    f.avg("open_rate").alias("open_rate"),
    f.avg("redemption_rate").alias("redemption_rate"),
    f.avg("download_rate").alias("download_rate"),
    f.sum("total_downloads").alias("total_downloads"),
    f.sum("total_redemptions_visits").alias("total_redemptions_visits")
        
# Manual Upstream Calculations for Closed Loop Campaigns
).withColumn(
        "iROAS",
        f.col("sales_uplift_total") / f.col("camp_cost")
    ).withColumn(
        "aROAS",
        f.col("sales_test_total") / f.col("camp_cost")
    ).withColumn(
        "sales_uplift_pct",
        f.col("sales_uplift_total") / f.col("sales_cont_total")
    ).withColumn(
        "hhpen_uplift_pct",
        f.col("hhpen_uplift_total") / f.col("hhpen_cont_total")
    ).withColumn(
        "visits_uplift_pct",
        f.col("visits_uplift_total") / f.col("visits_cont_total")
    ).withColumn(
        "units_uplift_pct",
        f.col("units_uplift_total") / f.col("units_cont_total")
    ).withColumn(
        "average_gift_card_load_amount",
        f.col("sales_test_total") / f.col("units_test_total")
    ).withColumn(
      "business_line",
      f.lit("Gift")
    )

media_history_v2_ctv_audio_agg.display()

In [0]:
# only calculate abs #s for NEW audio and ctv to save cluster compute

final_result_audio_ctv = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_ctv_audio').withColumnRenamed("kpm_duplicated_id", "kpm_project_id")
final_result_audio_ctv.display()

media_history_v2_ctv_audio_agg = media_history_v2_ctv_audio_agg.join(final_result_audio_ctv.select("kpm_project_id"), on = "kpm_project_id", how = 'left_anti')
media_history_v2_ctv_audio_agg.display()

#### Absolute iROAS Automation for Audio/CTV

In [0]:
data = [ 
    (800000202496, 168561, 'AUDIO'),
    (800000201991, 168561, 'AUDIO'),
    (800000201992, 168561, 'AUDIO'),
    (800000202495, 168561, 'AUDIO'),

    (800000216512, 173779, 'AUDIO'),
    (800000211929, 173779, 'AUDIO'),
    (800000216516, 173779, 'AUDIO'),
    (800000211934, 173779, 'AUDIO'),
    (800000228609, 173779, 'AUDIO'),
    (800000228610, 173779, 'AUDIO'),
    (800000228612, 173779, 'AUDIO'),
    (800000228611, 173779, 'AUDIO'),


]

schema = ["coupon_barcode", "campaign_id", "campaign_type"]
coupon_df = spark.createDataFrame(data, schema)
coupon_df.display()

campaign_ids = coupon_df.select('campaign_id').distinct().rdd.flatMap(lambda x: x).collect()
print(campaign_ids)

In [0]:
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')
mmci.filter(f.col("kpm_duplicated_id") == 168463).display()



In [0]:
# Hard coding the ID for easter audio, since it is under a diff. id in mmci
mmci

mmci = mmci.withColumn(
    "kpm_duplicated_id",
    f.when(f.col("kpm_project_id") == 168561, f.lit(168561))
     .otherwise(f.col("kpm_duplicated_id"))
)

mmci.filter(f.col("kpm_duplicated_id") == 168561).display()

# Working cost calc
mmci_audio_ctv_working_cost = (mmci.filter(f.col('kpm_project_id').isin(campaign_ids))
    .withColumn('multiplier',
         f.when(f.col('channel') == 'CTV', f.lit(0.62))
         .when(f.col('channel') == 'Programmatic Audio', f.lit(0.741))
         .otherwise(f.lit(0))
    )
    .withColumn('working_cost', (f.col('TOT_COST') * f.col('multiplier'))) 
    .groupBy('KPM_PROJECT_ID')
    .agg(
        f.max('CAMP_START_DATE').alias('camp_start_date'),
        f.max('CAMP_END_DATE').alias('camp_end_date'),
        f.sum('TOT_COST').alias('camp_cost_mmci'),
        f.sum('working_cost').alias('working_cost'),
    )
)

mmci_audio_ctv_working_cost = mmci_audio_ctv_working_cost.withColumnRenamed('KPM_DUPLICATED_ID', 'campaign_id')
mmci_audio_ctv_working_cost.display()

In [0]:
# Pulling coupons from Excel / Temp DF
coupon_barcodes_agg = (coupon_df
    .withColumn('coupon_barcode', f.col('coupon_barcode').cast('string'))
    .withColumn('redemption_barcode', f.lpad(f.col('coupon_barcode'), 13, '0'))
    .groupBy('campaign_id').agg(
        f.collect_set('coupon_barcode').alias('coupon_barcodes'),
        f.collect_set('redemption_barcode').alias('redemption_barcodes')
    )
    .withColumnRenamed('campaign_id', 'kpm_duplicated_id')
)

mmci_audio_ctv_working_cost_barcodes = coupon_barcodes_agg.join(
    mmci_audio_ctv_working_cost.select("KPM_PROJECT_ID", "camp_cost_mmci", "working_cost").withColumnRenamed('KPM_PROJECT_ID', 'kpm_duplicated_id'),
    'kpm_duplicated_id', 'inner'
)
display(mmci_audio_ctv_working_cost_barcodes)

In [0]:
# Join mmci (data already exists in dashboard revamped) with mmoi barcode/date information
kpf_dashboard_revamped_mmoi_audio_ctv_no_target = (media_history_v2_ctv_audio_agg
    .join(mmci_audio_ctv_working_cost_barcodes, f.col('kpm_duplicated_id') == f.col('KPM_PROJECT_ID'), how='inner')
    .drop('KPM_PROJECT_ID')
)

mmci_audio_ctv = mmci.filter((f.col("CHANNEL").isin("Programmatic Audio", "CTV")))

kpf_dashboard_revamped_mmoi_audio_ctv = kpf_dashboard_revamped_mmoi_audio_ctv_no_target.join(mmci_audio_ctv.select("KPM_DUPLICATED_ID", "CHANNEL", "TARGET_ID"), on = 'kpm_duplicated_id', how = 'left')
kpf_dashboard_revamped_mmoi_audio_ctv.display()

In [0]:
th_filter = (target_history
    .filter(f.col('test_control_id') == '1') # Filter early to reduce volume
    .withColumnRenamed('HSHD_CODE', 'ehhn')
    .join(
        f.broadcast(kpf_dashboard_revamped_mmoi_audio_ctv.select('kpm_duplicated_id', 'project_name', 'TARGET_ID')),
        on='TARGET_ID', 
        how='inner' 
    )
)

th_filter.display()

In [0]:
# Prepare necessary Points Detail - Faster Ver

points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
points_detail_target_history = (broadcast(th_filter)
    .join(points_detail, on='ehhn', how='inner'))

# Select only columns needed from mmoi_mhtv_mmci_sse
kpf_dashboard_revamped_mmoi_audio_ctv_subset = (kpf_dashboard_revamped_mmoi_audio_ctv
    .select(
        'kpm_duplicated_id', 
        'coupon_barcodes', 
        'redemption_barcodes', 
        'camp_start_date',
        'camp_end_date'
    )
)

# Convert transaction date to datetype 
points_detail_target_history = points_detail_target_history.withColumn(
    'trn_dt', f.to_date('trn_dt', 'yyyyMMdd')
)

# Join points detail + target history with dashboard and metadata (iroas, aroas, barcodes, camp info)
points_detail_target_history_offer_info = (
    points_detail_target_history
    .join(
        f.broadcast(kpf_dashboard_revamped_mmoi_audio_ctv_subset),
        on="kpm_duplicated_id",
        how="inner"
    )
    .filter(
        f.array_contains(f.col('coupon_barcodes'), f.col('offer')) |
        f.array_contains(f.col('redemption_barcodes'), f.col('offer'))
    )
)

points_detail_target_history_offer_info = points_detail_target_history_offer_info.dropDuplicates()
points_detail_target_history_offer_info.display()

In [0]:
# Filter to only transaction date is between effective and expiry date
points_detail_target_history_filtered_offer_info = (
    points_detail_target_history_offer_info
    .filter(
        (f.col('trn_dt') >= f.to_date(f.col('camp_start_date'), 'yyyyMMdd')) &
        (f.col('trn_dt') <= f.to_date(f.col('camp_end_date'), 'yyyyMMdd'))
    )
)

points_detail_target_history_filtered_offer_info.display()

In [0]:
#  Sum of points earned across all HHs per campaign
aggregated_df = (points_detail_target_history_filtered_offer_info
    .groupBy('kpm_duplicated_id', 'project_name')
    .agg(
        f.count('ehhn').alias('total_hhs'), 
        f.approx_count_distinct('ehhn').alias('distinct_ehhn'), 
        f.sum('points_earned').alias('total_points_earned')
    )
)

aggregated_df.display()

In [0]:
# Calculations (from Shumaila's code)
final_result_audio_ctv_df = (
    kpf_dashboard_revamped_mmoi_audio_ctv
    .join(aggregated_df, on=['kpm_duplicated_id', 'project_name'], how='left')
    .withColumn('cost_total_points_earned', f.round(f.col('total_points_earned') * 0.01, 2).cast('double'))
    .withColumn('cost_total_points_redemeed', f.round(f.col('total_points_earned') * 0.016 * 0.5887, 2).cast('double'))
    .withColumn('adj_sales_uplift', f.col("sales_uplift_total").cast('double'))
    .withColumn('adj_sales_total', f.col("sales_test_total").cast('double'))
    .withColumn('camp_cost', f.round(f.col("camp_cost").cast('double'), 2))
    .withColumn('working_cost', f.round(f.col("working_cost").cast('double'), 2))
    .withColumn('adj_total_cost', f.round((f.col('cost_total_points_earned') + f.col('camp_cost')).cast('double'), 2))
    .withColumn('abs_total_cost', f.round((f.col('cost_total_points_earned') + f.col('working_cost')).cast('double'), 2))
    # Added Redemption Cost explicitly to better match Shumaila's #s (same as cost_total_points_earned)
    .withColumn('redemption_cost', f.col('abs_total_cost') - f.col('working_cost'))
    .withColumn('abs_sales_uplift_earned', f.round((f.col('adj_sales_uplift') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('abs_sales_test_earned', f.round((f.col('adj_sales_total') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('adj_iroas', f.round((f.col('adj_sales_uplift') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_iroas', f.round((f.col('abs_sales_uplift_earned') / f.col('abs_total_cost')).cast('double'), 2))
    .withColumn('adj_aroas', f.round((f.col('adj_sales_total') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_aroas', f.round((f.col('abs_sales_test_earned') / f.col('abs_total_cost')).cast('double'), 2))
)

# Display the final aggregated DataFrame
final_result_audio_ctv_df.display()

In [0]:
final_result_audio_ctv_df.coalesce(1).write.mode("append").parquet(
    'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_ctv_audio'
)